# 02 - Direct vs two-stage의 정렬 보존 toy

**학습 목표**: 같은 미세조정 budget에서 이미 multimodal alignment를 가진 초기값이 왜 더 빠르게 높은 점수에 도달할 수 있는지 설명용 learning curve로 비교합니다. 논문의 73.3/60.2 결과를 fit한 모델이 아닙니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import math

def toy_score(progress, initial_alignment, learning_rate, ceiling=0.82):
    return ceiling - (ceiling - initial_alignment) * math.exp(-learning_rate * progress)

rows = []
for step in range(0, 11):
    direct = toy_score(step, initial_alignment=0.62, learning_rate=0.28)
    two_stage = toy_score(step, initial_alignment=0.24, learning_rate=0.16)
    rows.append((step, direct, two_stage))
    print(f'step={step:2d} direct={direct:0.3f} two-stage={two_stage:0.3f}')
assert rows[-1][1] > rows[-1][2]

In [ ]:
def area_under_learning_curve(rows, column):
    return sum(row[column] for row in rows) / len(rows)

print('mean score over fixed budget')
print(' direct   :', round(area_under_learning_curve(rows, 1), 3))
print(' two-stage:', round(area_under_learning_curve(rows, 2), 3))

# progress를 크게 늘리면 gap이 줄지만 같은 ceiling이라는 것은 이 toy의 가정일 뿐입니다.
for step in (10, 30, 100):
    print(step, round(toy_score(step, 0.62, 0.28), 3), round(toy_score(step, 0.24, 0.16), 3))

논문이 직접 입증한 것은 제한된 2M samples/1 epoch에서 direct가 더 좋다는 점입니다. 두 경로의 asymptotic ceiling이 같다는 설명은 저자의 가설이며 이 toy의 `ceiling`도 가정입니다.